# Génération de recommandations — Système de recommandation

Objectif : à partir du modèle SVD entraîné, générer les N films recommandés pour un utilisateur donné — le livrable concret du projet.

In [1]:
import pandas as pd
import numpy as np
import joblib

train_ratings = pd.read_csv('../data/processed/train_ratings.csv')
movies = pd.read_csv('../data/movies.csv')
svd_model = joblib.load('../outputs/models/svd_model.pkl')

print("Modèle SVD chargé")

Modèle SVD chargé


## Fonction de recommandation

Pour un utilisateur donné, on prédit sa note sur TOUS les films qu'il n'a pas encore vus, puis on retourne les N films avec la note prédite la plus haute.

In [2]:
def get_top_n_recommendations(user_id, n=10):
    all_movie_ids = movies['movieId'].unique()
    seen_movies = train_ratings[train_ratings['userId'] == user_id]['movieId'].unique()
    unseen_movies = [m for m in all_movie_ids if m not in seen_movies]

    predictions = []
    for movie_id in unseen_movies:
        pred = svd_model.predict(user_id, movie_id)
        predictions.append((movie_id, pred.est))

    predictions.sort(key=lambda x: x[1], reverse=True)
    top_n = predictions[:n]

    result = []
    for movie_id, pred_rating in top_n:
        title = movies[movies['movieId'] == movie_id]['title'].values[0]
        genres = movies[movies['movieId'] == movie_id]['genres'].values[0]
        result.append({'movieId': movie_id, 'title': title, 'genres': genres, 'predicted_rating': round(pred_rating, 2)})

    return pd.DataFrame(result)

## Exemple concret : recommandations pour un utilisateur

In [3]:
example_user = train_ratings['userId'].iloc[0]

print(f"=== Films déjà notés par l'utilisateur {example_user} (top 5 par note) ===")
user_history = train_ratings[train_ratings['userId'] == example_user].merge(movies, on='movieId')
user_history = user_history.sort_values('rating', ascending=False)[['title', 'genres', 'rating']].head(5)
print(user_history.to_string(index=False))

print(f"\n=== Top 10 recommandations pour l'utilisateur {example_user} ===")
recommendations = get_top_n_recommendations(example_user, n=10)
print(recommendations.to_string(index=False))

=== Films déjà notés par l'utilisateur 1 (top 5 par note) ===
                          title                                  genres  rating
Bedknobs and Broomsticks (1971)              Adventure|Children|Musical     5.0
     Princess Bride, The (1987) Action|Adventure|Comedy|Fantasy|Romance     5.0
              Highlander (1986)                Action|Adventure|Fantasy     5.0
               SLC Punk! (1998)                            Comedy|Drama     5.0
                 Henry V (1989)                Action|Drama|Romance|War     5.0

=== Top 10 recommandations pour l'utilisateur 1 ===
 movieId                                                                       title                      genres  predicted_rating
     260                                   Star Wars: Episode IV - A New Hope (1977)     Action|Adventure|Sci-Fi               5.0
     293              Léon: The Professional (a.k.a. The Professional) (Léon) (1994) Action|Crime|Drama|Thriller               5.0
     296    

## Vérification qualitative : cohérence genre-recommandation

On vérifie si les genres recommandés sont cohérents avec les genres préférés de l'utilisateur (un signal de bon sens, en complément du RMSE).

In [4]:
from collections import Counter

# Genres préférés (films notés >= 4) de l'utilisateur
liked_movies = train_ratings[(train_ratings['userId'] == example_user) & (train_ratings['rating'] >= 4)]
liked_movies = liked_movies.merge(movies, on='movieId')
liked_genres = Counter()
for genres_str in liked_movies['genres']:
    liked_genres.update(genres_str.split('|'))

print("Genres préférés de l'utilisateur (films notés >= 4) :")
print(liked_genres.most_common(5))

recommended_genres = Counter()
for genres_str in recommendations['genres']:
    recommended_genres.update(genres_str.split('|'))

print("\nGenres des films recommandés :")
print(recommended_genres.most_common(5))

Genres préférés de l'utilisateur (films notés >= 4) :
[('Action', 63), ('Adventure', 58), ('Comedy', 53), ('Drama', 49), ('Thriller', 34)]

Genres des films recommandés :
[('Crime', 5), ('Drama', 5), ('Comedy', 4), ('Action', 3), ('Thriller', 3)]


## Precision@K — évaluation de la qualité des recommandations

Au-delà du RMSE (précision de la note prédite), on évalue si les films effectivement recommandés (top-K) correspondent à des films que l'utilisateur a vraiment aimés dans le test set.

In [5]:
test_ratings = pd.read_csv('../data/processed/test_ratings.csv')

def precision_at_k(user_id, k=10, threshold=4.0):
    # Films que l'utilisateur a réellement aimés dans le test set
    liked_in_test = set(test_ratings[
        (test_ratings['userId'] == user_id) & (test_ratings['rating'] >= threshold)
    ]['movieId'])

    if len(liked_in_test) == 0:
        return None  # pas de vérité terrain disponible pour cet utilisateur

    recommended = get_top_n_recommendations(user_id, n=k)
    recommended_ids = set(recommended['movieId'])

    hits = len(recommended_ids & liked_in_test)
    return hits / k

# Évaluation sur un échantillon d'utilisateurs (pour le temps de calcul)
sample_users = train_ratings['userId'].unique()[:50]
precisions = []
for user_id in sample_users:
    p = precision_at_k(user_id, k=10)
    if p is not None:
        precisions.append(p)

print(f"Precision@10 moyenne sur {len(precisions)} utilisateurs évalués : {np.mean(precisions):.4f}")
print(f"(sur 10 films recommandés, en moyenne {np.mean(precisions)*10:.1f} correspondent à un film que l'utilisateur a vraiment aimé dans le test set)")

Precision@10 moyenne sur 49 utilisateurs évalués : 0.0306
(sur 10 films recommandés, en moyenne 0.3 correspondent à un film que l'utilisateur a vraiment aimé dans le test set)
